In [ ]:
from reachy_sdk import ReachySDK

reachy = ReachySDK(host="10.22.128.166")

print(reachy.joints.keys())

In [ ]:
import cv2
import numpy as np

# --------------------------------------------------
# TFLITE / LITERT INTERPRETER
# --------------------------------------------------

try:
    from tflite_runtime.interpreter import Interpreter
    print("Using tflite_runtime")
except ImportError:
    try:
        from ai_edge_litert.interpreter import Interpreter
        print("Using ai_edge_litert")
    except ImportError:
        import tensorflow as tf
        Interpreter = tf.lite.Interpreter
        print("Using TensorFlow Lite")

# --------------------------------------------------
# CONFIGURATION
# --------------------------------------------------

MODEL_PATH = "best.tflite"

CLASS_NAMES = [
    "cube",
    "cylinder",
    "empty"
]

INPUT_SIZE = 320

# Start fairly low for initial testing.
CONF_THRESHOLD = 0.30
NMS_THRESHOLD = 0.45


# --------------------------------------------------
# LOAD MODEL
# --------------------------------------------------

interpreter = Interpreter(model_path=MODEL_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

input_index = input_details[0]["index"]
output_index = output_details[0]["index"]

print("Model loaded successfully")
print("Input shape :", input_details[0]["shape"])
print("Input dtype :", input_details[0]["dtype"])
print("Output shape:", output_details[0]["shape"])
print("Output dtype:", output_details[0]["dtype"])


# --------------------------------------------------
# YOLO DETECTION
# --------------------------------------------------

def detect(frame):

    original_h, original_w = frame.shape[:2]

    # frame arrives here as BGR.
    # YOLO expects RGB.
    image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Resize to model input size.
    image = cv2.resize(
        image,
        (INPUT_SIZE, INPUT_SIZE)
    )

    # Convert to float32 and normalize 0-1.
    image = image.astype(np.float32) / 255.0

    # HWC -> CHW
    #
    # This is IMPORTANT for your model because its
    # input tensor is [1, 3, 320, 320], not
    # [1, 320, 320, 3].
    image = np.transpose(image, (2, 0, 1))

    # CHW -> NCHW
    image = np.expand_dims(image, axis=0)

    interpreter.set_tensor(
        input_index,
        image
    )

    interpreter.invoke()

    output = interpreter.get_tensor(
        output_index
    )

    # Your output is:
    # [1, 7, 2100]
    #
    # Convert to:
    # [2100, 7]
    predictions = output[0].T

    boxes = []
    confidences = []
    class_ids = []

    for prediction in predictions:

        # YOLO box
        x_center = prediction[0]
        y_center = prediction[1]
        width = prediction[2]
        height = prediction[3]

        # Remaining 3 values are class scores
        class_scores = prediction[4:]

        class_id = int(np.argmax(class_scores))
        confidence = float(class_scores[class_id])

        if confidence < CONF_THRESHOLD:
            continue

        # Convert normalized coordinates to
        # original Reachy camera coordinates.
        x_center *= original_w
        y_center *= original_h
        width *= original_w
        height *= original_h

        x = int(x_center - width / 2)
        y = int(y_center - height / 2)

        w = int(width)
        h = int(height)

        boxes.append([x, y, w, h])
        confidences.append(confidence)
        class_ids.append(class_id)

    # --------------------------------------------------
    # NON-MAXIMUM SUPPRESSION
    # --------------------------------------------------

    detections = []

    if len(boxes) == 0:
        return detections

    indices = cv2.dnn.NMSBoxes(
        boxes,
        confidences,
        CONF_THRESHOLD,
        NMS_THRESHOLD
    )

    for i in indices:

        # Handles different OpenCV return formats.
        if isinstance(i, (list, tuple, np.ndarray)):
            i = int(np.array(i).flatten()[0])
        else:
            i = int(i)

        x, y, w, h = boxes[i]

        detections.append({
            "class_id": class_ids[i],
            "label": CLASS_NAMES[class_ids[i]],
            "confidence": confidences[i],
            "box": (x, y, w, h)
        })

    return detections


# --------------------------------------------------
# REACHY CAMERA LOOP
# --------------------------------------------------

while True:

    frame = reachy.right_camera.last_frame

    if frame is None:
        continue

    # Reachy image -> NumPy
    frame = np.array(frame)

    # Reachy camera provides RGB.
    # OpenCV uses BGR.
    frame = cv2.cvtColor(
        frame,
        cv2.COLOR_RGB2BGR
    )

    detections = detect(frame)

    # --------------------------------------------------
    # DRAW DETECTIONS
    # --------------------------------------------------

    for detection in detections:

        x, y, w, h = detection["box"]

        label = detection["label"]
        confidence = detection["confidence"]

        x2 = x + w
        y2 = y + h

        cv2.rectangle(
            frame,
            (x, y),
            (x2, y2),
            (0, 255, 0),
            2
        )

        text = f"{label}: {confidence:.2f}"

        text_y = y - 10 if y > 20 else y + 20

        cv2.putText(
            frame,
            text,
            (x, text_y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 255, 0),
            2
        )

        print(
            f"Detected: {label} "
            f"confidence={confidence:.2f}"
        )

    # --------------------------------------------------
    # DISPLAY
    # --------------------------------------------------

    cv2.imshow(
        "Reachy YOLO Detection",
        frame
    )

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


cv2.destroyAllWindows()